# Lab 6.2 &mdash; Retrieval as a Tool the Agent Chooses

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Wrap the retriever in a <code>@tool</code> &mdash; and write the description the model actually reads
- Offer it alongside the ledger tool with <code>bind_tools</code>, and let the model decide whether to retrieve
- Price always-retrieve: the tokens, and the noise it puts next to the real context
- Rewrite the user's words into the corpus's vocabulary

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a chunked
> `Document`, a Chroma collection, a bound tool, a compiled graph, a parser), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code in
> front of the sandbox model; that is the part worth watching. The score line is feedback, not a
> grade.

> **Builds directly on Lab 6.1's retriever.** Same splitter, same embeddings, same
> Chroma collection. What changes is who decides when it runs, and what it is asked.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# The same payment exceptions the other modules work on. Nothing here is real data.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

print(f"{len(LEDGER)} payments loaded")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents about the same payments. Read 3.2: the rule and the exception
# that qualifies it are adjacent sentences, which is the whole of Lab 6.1's first lesson. Note
# also what is NOT here -- nothing mentions FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- the embedding model (nothing to fill in)
# The sandbox has no egress, and chromadb's DEFAULT embedding function downloads about 80 MB
# of ONNX model the first time it is called. So this module brings its own: one hashed bucket
# per meaningful word, normalised to unit length. It is arithmetic rather than learning, which
# is the point -- it runs offline, it is deterministic, and you can read every line of it.
#
# What it CAN do: score two texts by the words they share. What it CANNOT do: match meaning
# with no words in common. Lab 6.2 is about living with exactly that.
import re, math, hashlib
from langchain_core.embeddings import Embeddings

STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def content_words(text: str) -> list:
    """The words worth indexing: lower-cased, no punctuation, no stop words."""
    return [w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1]


class LabEmbeddings(Embeddings):
    """A tiny embedding model you can read. Same interface as any other LangChain embedding."""

    dim = 1024                      # enough buckets that two different words rarely collide

    def _vector(self, text: str) -> list:
        vec = [0.0] * self.dim
        for word in content_words(text):
            bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % self.dim
            vec[bucket] += 1.0
        length = math.sqrt(sum(x * x for x in vec)) or 1.0
        return [x / length for x in vec]      # unit length, so cosine is just a dot product

    def embed_documents(self, texts: list) -> list:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list:
        return self._vector(text)


print("embeddings:", LabEmbeddings.dim, "dimensions, offline, deterministic")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
# Exactly what you built in Lab 6.1: split on headings, index in Chroma, search with a floor.
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma

FLOOR = 0.20            # the similarity a chunk must clear to be used at all (Lab 6.1)

def section_chunks() -> list:
    """One Document per '##' section, with the heading kept in the text and in the metadata."""
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("##", "section")],
                                          strip_headers=False)
    out = []
    for name, text in DOCS.items():
        for chunk in splitter.split_text(text):
            chunk.metadata["source"] = name
            out.append(chunk)
    return out


_store = None
def store():
    """The Chroma collection, built once, on first use."""
    global _store
    if _store is None:
        chunks = section_chunks()
        _store = Chroma(collection_name="module6-corpus",
                        embedding_function=LabEmbeddings(),
                        persist_directory=os.path.join(WORK, "chroma"),
                        collection_configuration={"hnsw": {"space": "cosine"}})
        # ids derived from the chunk, so re-running this notebook updates instead of duplicating
        _store.add_documents(chunks, ids=[f"{c.metadata['source']}#{c.metadata['section']}"
                                          for c in chunks])
    return _store


def search(query: str, k: int = 4, floor: float = 0.0, where: dict | None = None) -> list:
    """Top-k from the store as plain dicts, with anything below `floor` dropped."""
    hits = store().similarity_search_with_score(query, k=k, filter=where)
    out = []
    for doc, distance in hits:
        similarity = 1.0 - distance         # cosine space: 1.0 identical, 0.0 nothing in common
        if similarity >= floor:
            out.append({"score": round(similarity, 3), "text": doc.page_content,
                        "source": doc.metadata["source"], "section": doc.metadata["section"]})
    return out


print(f"index ready: {len(store().get()['ids'])} chunks")

## Concept

A pipeline retrieves once, always, with the user's exact words. That is one decision, made at
build time, applied to every question.

An agent makes three decisions per question, and this lab builds the first two:

- **whether** to retrieve &mdash; some questions are answered by the ledger, by the conversation,
  or by arithmetic
- **what to ask for** &mdash; users write in their words, corpora in the organisation's

The third, *whether to ask again*, is Lab 6.3.

The mechanism for the first one is not a rule you write. It is a **tool description**: the model
chooses between the tools you bind, and the description is all it has to choose on.

## Section 1 &mdash; Wrap retrieval in a tool

`@tool` turns a function into something a model can call. It takes the **name** from the function,
the **argument schema** from the type hints, and the **description from the docstring**.

That docstring is the whole interface. Measured on this sandbox: writing the descriptions
properly moved first-tool accuracy from **2/5 to 5/5** &mdash; same model, same functions.

In [ ]:
from langchain_core.tools import tool

def retrieve_text(query: str) -> str:
    """The retrieval itself: search the corpus and format what came back. Nothing to fill in."""
    hits = search(query, k=3, floor=FLOOR)
    if not hits:
        return "nothing in the operating documents cleared the relevance floor for that query"
    return "\n\n".join(f"[{h['source']} #{h['section']}] {h['text']}" for h in hits)


@tool
def search_operating_docs(query: str) -> str:
    """BLANK"""
    # TODO: replace that docstring. It is the ONLY thing the model reads when it decides
    # whether to call this tool. Say what the corpus contains (the payments operating runbook
    # and the escalation policy), what a good query looks like, and -- the part people skip --
    # what this tool is NOT for. Use the word "not" when you say it.
    return retrieve_text(query)


@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record (amount, currency, counterparty, status, reason code) for ONE
    payment reference such as 'PMT-1003'. Use when the question names a specific payment. This
    reads the ledger only -- it is not a search over documents, procedures or policy.
    """
    record = LEDGER.get(ref)
    return json.dumps({"ref": ref, **record}) if record else f"no payment found for {ref!r}"

In [ ]:
# --- Self-check: Section 1   (the tool OBJECTS, and one real retrieval -- no model)
def _desc(t) -> str:
    d = (t.description or "").strip()
    if d == "BLANK" or not d:
        raise NameError(f"{t.name} still has the placeholder docstring")
    return d

check("@tool took the name from the function",
      lambda: search_operating_docs.name == "search_operating_docs")
check("the argument schema was inferred from the type hint",
      lambda: "query" in search_operating_docs.args)
check("the description is a description, not a label",
      lambda: len(_desc(search_operating_docs)) > 80,
      "'Searches documents.' tells the model nothing it could not guess from the name")
check("it names what is in the corpus",
      lambda: any(w in _desc(search_operating_docs).lower()
                  for w in ("runbook", "escalation", "operating")),
      "the model cannot guess which documents you indexed")
check("it says what the tool is NOT for",
      lambda: "not" in _desc(search_operating_docs).lower(),
      "two tools that both sound like 'looks things up' is how the wrong one gets called")
check("calling the tool really retrieves",
      lambda: "3.2" in search_operating_docs.invoke({"query": "approval above USD 500,000"}))
check("and it says so plainly when nothing clears the floor",
      lambda: "nothing" in search_operating_docs.invoke(
          {"query": "FX hedging policy for JPY"}).lower(),
      "the floor from Lab 6.1, doing its job inside a tool")
check("the ledger tool is a different tool with a different boundary",
      lambda: "PMT-1003" in lookup_payment.invoke({"ref": "PMT-1003"}))

## Section 2 &mdash; Offer both, and let the model choose

`bind_tools` returns a model that may answer with a **tool call** instead of prose. Give it both
tools and *whether to retrieve* stops being a rule you maintain and becomes a choice it makes
per question &mdash; on the strength of the descriptions you just wrote.

In [ ]:
def agent_tools() -> list:
    """The tool OBJECTS this agent may choose between. bind_tools wants objects, not names."""
    return BLANK          # TODO: which tools may it call?


def bound_model():
    """A model allowed to answer with a tool call. Used by the live cells below."""
    return get_llm().bind_tools(agent_tools())

In [ ]:
# --- Self-check: Section 2   (the tool list -- building it needs no endpoint)
check("both tools are on offer",
      lambda: {t.name for t in agent_tools()} == {"lookup_payment", "search_operating_docs"})
check("bind_tools was given the objects, not their names",
      lambda: all(hasattr(t, "invoke") and hasattr(t, "name") for t in agent_tools()),
      "a list of strings binds nothing -- the model would be offered no tools at all")
check("every tool it may choose carries a real description",
      lambda: all(len(_desc(t)) > 80 for t in agent_tools()),
      "the choice the model makes is made entirely out of these strings")

## Section 3 &mdash; What always-retrieve costs

Before the model gets a say, price the alternative. A pipeline retrieves for every question. Two
things go wrong, and the second does not show up on an invoice: tokens spent, and irrelevant
policy prose sitting next to the context the answer was actually in.

The keyword rule below is the version you could write by hand. It is here as a *baseline* &mdash;
it works on the eight questions someone thought of, and the live cell asks whether the model
does better without one.

In [ ]:
QUESTIONS = [
    # (question, does it need the corpus?)
    ("What approval does a payment above USD 500,000 need?",       True),
    ("What happens when a payment comes back INVALID_IBAN?",       True),
    ("Who decides on a payment held for sanctions review?",        True),
    ("How long before an unanswered approval escalates?",          True),
    ("What is 990,000 minus 500,000?",                             False),
    ("Calculate the difference between the amount and the limit.", False),
    ("Summarise what we just agreed.",                             False),
    ("What did I ask you a moment ago?",                           False),
]

CONVERSATION_HINTS = ("we just", "you said", "a moment ago", "earlier you", "we agreed",
                      "summarise what we", "recap")
ARITHMETIC_HINTS   = ("plus", "minus", "times", "calculate", "subtract", "difference between",
                      "how much is")

def needs_corpus(question: str) -> bool:
    """The hand-written baseline: two kinds of question do not need the corpus."""
    low = (question or "").lower()
    return not any(h in low for h in CONVERSATION_HINTS + ARITHMETIC_HINTS)


def retrieved_tokens(question: str, always: bool) -> int:
    """Roughly what retrieval put into the context for this question."""
    if not always and not needs_corpus(question):
        return 0
    return sum(len(r["text"]) // 4 for r in search(question, k=4))

In [ ]:
# --- Self-check: Section 3   (counting, over the real index -- no model)
check("the baseline gets all eight questions right",
      lambda: all(needs_corpus(q) is expected for q, expected in QUESTIONS))
check("half the set needs no corpus at all",
      lambda: sum(1 for _, e in QUESTIONS if not e) == 4,
      "that fraction is the whole argument -- a pipeline retrieves for all eight")
check("always-retrieve spends tokens on questions that needed nothing",
      lambda: sum(retrieved_tokens(q, always=True) for q, e in QUESTIONS if not e) > 0)
check("deciding first spends none",
      lambda: sum(retrieved_tokens(q, always=False) for q, e in QUESTIONS if not e) == 0)
check("and the corpus questions are retrieved identically either way",
      lambda: all(retrieved_tokens(q, True) == retrieved_tokens(q, False)
                  for q, e in QUESTIONS if e),
      "deciding is not retrieving less well -- it is retrieving less often")
check("the noise is the part that never shows on the invoice",
      lambda: len(search("Summarise what we just agreed.", k=4)) == 4,
      "four chunks of policy prose, competing with the conversation the answer is actually in")

def _cost():
    always = sum(retrieved_tokens(q, True) for q, _ in QUESTIONS)
    decide = sum(retrieved_tokens(q, False) for q, _ in QUESTIONS)
    print(f"  always retrieve : {always:>5} retrieved tokens")
    print(f"  decide first    : {decide:>5} retrieved tokens")
guard(_cost)

## Section 4 &mdash; Ask in the corpus's words

Users describe their situation. Documents describe the organisation's rules. Translating between
them is the cheapest retrieval improvement there is, because it changes nothing about the index.

In [ ]:
# What people say -> what the documents call it
VOCAB = {
    "bounce":           "INSUFFICIENT_FUNDS retry",
    "bounced":          "INSUFFICIENT_FUNDS retry",
    "push it through":  "release Treasury approval",
    "push through":     "release Treasury approval",
    "wrong account":    "INVALID_IBAN beneficiary originator",
    "bad iban":         "INVALID_IBAN beneficiary originator",
    "on hold":          "SANCTIONS_REVIEW Compliance",
    "stuck":            "SANCTIONS_REVIEW Compliance",
    "chase":            "escalate approver Treasury lead",
    "our own entities": "intra-group transfers",
}

def rewrite(question: str) -> str:
    """The query the agent actually sends."""
    extra = [v for k, v in VOCAB.items() if k in (question or "").lower()]
    if not extra:
        return question
    # TODO: build the query out of `question` and `extra`. One of them alone is wrong:
    # the table cannot cover every phrasing, and the user's words do not match the documents.
    return BLANK

In [ ]:
# --- Self-check: Section 4   (retrieval quality, measured -- no model)
VAGUE = [
    ("Why did this one bounce, and do we try again?",           "3.1"),
    ("The client gave us the wrong account number. Now what?",  "3.3"),
    ("It is stuck. Who decides?",                               "3.4"),
    ("Can we push a big one through between our own entities?", "3.2"),
]

def best_section(query: str):
    hits = search(query, k=1)
    return hits[0]["section"] if hits else None

check("the rewrite keeps the user's own words",
      lambda: rewrite("Why did this one bounce?").startswith("Why did this one bounce?"),
      "the table cannot cover everything; dropping the original loses whatever it missed")
check("and adds the corpus vocabulary they imply",
      lambda: "INSUFFICIENT_FUNDS" in rewrite("Why did this one bounce?"))
check("a question with no match is passed through unchanged",
      lambda: rewrite("Who approves a release?") == "Who approves a release?")
check("RAW, these four vague questions land on the wrong section",
      lambda: sum(1 for q, want in VAGUE if (best_section(q) or "").startswith(want)) <= 1)
check("rewritten, they all land on the right one",
      lambda: all((best_section(rewrite(q)) or "").startswith(want) for q, want in VAGUE),
      "same index, same embeddings, same k -- the only change is who wrote the query")

def _rewrites():
    for q, want in VAGUE:
        print(f"  {q}")
        print(f"      raw       -> {best_section(q)}")
        print(f"      rewritten -> {best_section(rewrite(q))}   (want {want})")
guard(_rewrites)

## Run it for real &mdash; part 1: does the description do the work?

Two bindings of the same two functions. One arm has your descriptions; the other has what a
rushed codebase actually looks like. Same model, same questions. Watch which tool it reaches for.

In [ ]:
if llm_ready():
    def _description_ab():
        from langchain_core.tools import StructuredTool

        def _ledger(ref: str) -> str:
            return lookup_payment.invoke({"ref": ref})

        def _docs(query: str) -> str:
            return retrieve_text(query)

        # the same two functions, behind the descriptions a rushed codebase actually ships
        vague = [
            StructuredTool.from_function(_ledger, name="tool_a", description="Gets data."),
            StructuredTool.from_function(_docs,   name="tool_b", description="Looks things up."),
        ]
        probes = [
            ("What approval does a payment above USD 500,000 need?", "docs"),
            ("What is the status of PMT-1003?",                      "ledger"),
            ("Who decides on a payment held for sanctions review?",  "docs"),
            ("How much is PMT-1005 for?",                            "ledger"),
            ("How long before an unanswered approval escalates?",    "docs"),
        ]
        for label, tools in (("vague descriptions", vague), ("your descriptions", agent_tools())):
            right = 0
            for question, want in probes:
                calls = get_llm().bind_tools(tools).invoke(question).tool_calls
                picked = calls[0]["name"] if calls else "(no tool)"
                got = "ledger" if picked in ("tool_a", "lookup_payment") else \
                      "docs" if picked in ("tool_b", "search_operating_docs") else "?"
                right += got == want
                print(f"  [{label:18}] {question[:44]:46} -> {picked}")
            print(f"  [{label:18}] first-tool accuracy {right}/{len(probes)}\n")
    guard(_description_ab)

## Run it for real &mdash; part 2: let the model write the query

The lookup table is a stand-in. This is what you would actually ship, because no table survives
contact with real users.

In [ ]:
if llm_ready():
    def _model_rewrite():
        vocab_hint = ("The documents use terms like: INSUFFICIENT_FUNDS, INVALID_IBAN, "
                      "SANCTIONS_REVIEW, Treasury approval, intra-group transfer, escalation.")
        for q, want in VAGUE:
            query = ask(f"{vocab_hint}\n\nRewrite this into a search query using those terms. "
                        f"Reply with the query alone.\n\n{q}",
                        system="Reply with a search query and nothing else.")
            got = best_section((query or "").strip())
            flag = "ok  " if (got or "").startswith(want) else "MISS"
            print(f"  [{flag}] {q[:42]:44} -> {got}")
    guard(_model_rewrite)

### Read it

**The A/B.** Two identical functions behind two sets of strings. The equivalent experiment on this
sandbox in Module 1 &mdash; five tools, same model &mdash; moved first-tool accuracy from **2/5 to 5/5** on
the descriptions alone. Two tools is an easier problem than five, so expect a smaller gap here;
what you are watching for is *which* questions the vague arm gets wrong. It will be the ones where
both names sound equally plausible. If your own arm scores badly, read your description the way the
model does: does it say which questions belong to this tool, and which do not?

**The rewrites.** If the model's rewrites land as well as the lookup table's, you have something
that generalises to questions you never enumerated, at the cost of one model call before every
retrieval. That is a real trade, and Lab 6.5 is where you price it. Watch for the failure mode
too: a rewrite that invents a term the corpus does not contain retrieves *worse* than the raw
question. A query, like a tool description, can attract the wrong thing as easily as the right one.

In [ ]:
score()

## Your turn

1. `needs_corpus` is a keyword list, so it fails on any phrasing you did not think of. Write three
   questions that should not retrieve and that it gets wrong. What does that tell you about
   shipping the rule rather than the tool description?
2. Add a third tool that overlaps with `search_operating_docs` &mdash; say `search_escalation_policy`,
   scoped with the metadata filter from Lab 6.1. Now write both descriptions so the model can tell
   them apart, and re-run the A/B.
3. There is a third answer besides yes and no: *retrieve, but only if the first attempt at
   answering is thin*. Sketch it, and say what it costs in latency. That is Lab 6.3.